<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/modelamiento-optimizacion-redes/02_diseno_red_multiflujo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Diseño de red con flujo multiproducto (ILP)

**Modelamiento y Optimización en Redes de Comunicaciones** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Formular el problema de diseño de red con costos fijos y variables.
- Escribir restricciones de conservación de flujo para varias demandas.
- Resolver con programación lineal entera mixta (PuLP + CBC).
- Comparar enrutamiento divisible y no divisible (un solo camino).

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
!pip -q install pulp
import pulp, networkx as nx, matplotlib.pyplot as plt

## 1. Formulación
Se decide qué enlaces candidatos **instalar** ($y_e \in \{0,1\}$, costo fijo $f_e$) y cómo **enrutar** cada demanda $k$ ($x^k_{ij} \ge 0$, costo $c_e$ por Gbps):

$$\min \sum_e f_e y_e + \sum_k \sum_{(i,j)} c_{ij} x^k_{ij}$$

$$\sum_j x^k_{ij} - \sum_j x^k_{ji} = \begin{cases} d_k & i = s_k \\ -d_k & i = t_k \\ 0 & \text{otro caso}\end{cases} \qquad \forall k, i$$

$$\sum_k (x^k_{ij} + x^k_{ji}) \le C\, y_{ij} \qquad \forall e=(i,j)$$

In [ ]:
nodos = ['A', 'B', 'C', 'D', 'E', 'F']
# (u, v, costo fijo, costo por Gbps)
candidatos = [('A', 'B', 10, 1), ('A', 'C', 12, 2), ('B', 'C', 6, 1), ('B', 'D', 14, 2), ('C', 'E', 9, 1), ('D', 'E', 7, 1), ('D', 'F', 11, 2), ('E', 'F', 8, 1), ('C', 'D', 10, 1)]
demandas = {('A', 'F'): 8, ('B', 'E'): 5, ('A', 'D'): 4}  # Gbps
C = 12  # capacidad de cada enlace instalado (Gbps)

In [ ]:
def disenar(C=C, un_camino=False, dibujar=True):
    m = pulp.LpProblem("diseno", pulp.LpMinimize)
    arcos = [(u, v) for u, v, _, _ in candidatos] + [(v, u) for u, v, _, _ in candidatos]
    f = {(u, v): fc for u, v, fc, _ in candidatos}
    c = {(u, v): vc for u, v, _, vc in candidatos}
    arista = lambda a: a if a in f else (a[1], a[0])
    y = {e: pulp.LpVariable(f"y_{e[0]}{e[1]}", cat="Binary") for e in f}
    if un_camino:
        z = {(k, a): pulp.LpVariable(f"z_{k[0]}{k[1]}_{a[0]}{a[1]}", cat="Binary") for k in demandas for a in arcos}
        x = {(k, a): demandas[k] * z[k, a] for k in demandas for a in arcos}
    else:
        x = {(k, a): pulp.LpVariable(f"x_{k[0]}{k[1]}_{a[0]}{a[1]}", lowBound=0) for k in demandas for a in arcos}
    m += pulp.lpSum(f[e]*y[e] for e in y) + pulp.lpSum(c[arista(a)]*x[k, a] for k in demandas for a in arcos)
    for k, d in demandas.items():
        s, t = k
        for n in nodos:
            sale = pulp.lpSum(x[k, a] for a in arcos if a[0] == n)
            entra = pulp.lpSum(x[k, a] for a in arcos if a[1] == n)
            m += sale - entra == (d if n == s else -d if n == t else 0)
    for e in y:
        m += pulp.lpSum(x[k, e] + x[k, (e[1], e[0])] for k in demandas) <= C * y[e]
    m.solve(pulp.PULP_CBC_CMD(msg=0))
    print("Estado:", pulp.LpStatus[m.status], "| costo total =", pulp.value(m.objective))
    if dibujar:
        H = nx.Graph([(u, v) for u, v, _, _ in candidatos])
        pos = nx.spring_layout(H, seed=4)
        on = [e for e in y if y[e].value() > 0.5]
        nx.draw(H, pos, with_labels=True, node_color="#dcefe4", edge_color="#ccc", style="dashed")
        nx.draw_networkx_edges(H, pos, edgelist=on, width=3, edge_color="#0b6b45")
        plt.title("Enlaces instalados"); plt.show()
    return pulp.value(m.objective)

disenar()

## 2. Ejercicios
**Ejercicio 1.** Costo óptimo del diseño base → `resp_1`.

**Ejercicio 2.** Si los enlaces solo tienen **10 Gbps**, ¿cuál es el nuevo costo óptimo? → `resp_2`. ¿Qué enlaces adicionales se instalan?

**Ejercicio 3.** Con $C=12$, exija que cada demanda use **un solo camino** (`un_camino=True`) → `resp_3`. ¿Por qué el costo no puede ser menor que el del ejercicio 1?

In [ ]:
resp_1 = None
resp_2 = None
resp_3 = None

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 107.0, tol=1e-4)
verificar('Ejercicio 2', resp_2, 119.0, tol=1e-4)
verificar('Ejercicio 3', resp_3, 110.0, tol=1e-4)

**Ejercicio 4 (reto).** Agregue **protección 1+1**: cada demanda debe tener dos caminos disjuntos en enlaces. Modifique la formulación y discuta el aumento de costo.

**Para ir más lejos:** descargue una instancia real de [SNDlib](https://sndlib.put.poznan.pl/) y adapte el código.